Loading tools

In [6]:
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager", repos = "https://cloud.r-project.org")

BiocManager::install(version = "3.22")

BiocManager::install(c(
    "minfi",
    "IlluminaHumanMethylationEPICmanifest",
    "IlluminaHumanMethylationEPICanno.ilm10b4.hg19"
), update = FALSE, ask = FALSE)

ERROR: Error: Bioconductor version '3.22' requires R version '4.5'; use `version = '3.23'`
  with R version 4.6; see https://bioconductor.org/install


In [7]:
library(minfi)
packageVersion("minfi")

[1] ‘1.58.0’

This fails because the 88 IDAT files aren't all the same physical array size.

In [8]:
library(minfi)

targets <- read.csv(here::here("data", 'GSE189148_resting_sample_metadata.csv"))

rgSet <- read.metharray.exp(targets = targets)

#Error here: IDAT files have different array sizes

ERROR: Error in parse(text = input): <text>:3:40: unexpected INCOMPLETE_STRING
6: 
7: #Error here: IDAT files have different array sizes
                                          ^


In [10]:
library(illuminaio)

grn_files <- list.files(here::here("data", "GSE189148_idats"), pattern = "_Grn.idat.gz$", full.names = TRUE)

probe_counts <- sapply(grn_files, function(f) {
  idat <- readIDAT(f)
  idat$nSNPsRead
})

table(probe_counts)

probe_counts
1051815 1051943 
     23      65 

The two different probe counts (1,051,815 vs. 1,051,943) do not line up with allergy status; all three groups (control, single-food-allergic, multi-food-allergic) contain a mix of both. It's most likely a chip-batch/version artifact. 

In [14]:
targets <- read.csv(here::here("data", "GSE189148_resting_sample_metadata.csv"))

targets$probe_count <- sapply(paste0(targets$Basename, '_Grn.idat.gz'), function(f) {
  readIDAT(f)$nSNPsRead
})

table(targets$allergy_status, targets$probe_count) #Checking if the split has anything to do with allergic/healthy labels
print(fisher.test(matrix(c(3, 12, 6, 22), nrow = 2, byrow = TRUE)))

                      
                       1051815 1051943
  control                    3      12
  multi_food_allergic        2      13
  single_food_allergic       4       9


	Fisher's Exact Test for Count Data

data:  matrix(c(3, 12, 6, 22), nrow = 2, byrow = TRUE)
p-value = 1
alternative hypothesis: true odds ratio is not equal to 1
95 percent confidence interval:
 0.1259299 5.3005581
sample estimates:
odds ratio 
 0.9184902 



In [12]:
targets <- read.csv(here::here("data", "GSE189148_resting_sample_metadata.csv"))
rgSet <- read.metharray.exp(targets = targets, force = TRUE) #Just forcing through

In [16]:
saveRDS(rgSet, here::here("data", "rgSet_resting.rds"))

- 88 GSE189148 resting-state samples successfully loaded as an RGChannelSet despite an array-size mismatch across IDAT files (1,051,815 vs. 1,051,943 probes), confirmed unrelated to allergy status.
- Saved as rgSet_resting.rds ans and used in notebook 3 (quality control) going forward.